# Modelado de proteínas y docking
## Interacción GRIN2B – Ácido quinolínico (QUIN) con AlphaFold + AutoDock Vina
######Notebook elaborado por Laura Itzel Hernández Romero


**Objetivo:** estudiar *in silico* la posible interacción entre la subunidad NR2B (GRIN2B) del receptor NMDA y el ácido quinolínico (QUIN).

Flujo de trabajo completo:
1. Conocer cómo modelar una proteína
2. Descargar el modelo de GRIN2B desde la base de datos de AlphaFold.
3. Evaluar la calidad del modelo.  
4. Preparar la proteína y el ligando QUIN para el docking.  
5. Definir la caja o región de interacción de docking.  
6. Ejecutar AutoDock Vina.  
7. Visualizar el complejo proteína–ligando y analizar la afinidad.



## Consideración teórica
- El gen ** *GRIN2B* (NR2B)** (chr.12), codifica la proteína GluN2B, subunidad de los receptores NMDA, esencial en la transmisión glutamatérgica y la plasticidad sináptica.  
- El **ácido quinolínico (QUIN)** es un metabolito del triptófano que actúa como agonista excitotóxico del receptor NMDA. La activación excesiva de estos receptores por el QUIN por la unión del QUIN, lo activa y provoca la entrada excesiva de calcio en la neurona causando daño neuronal y disfunción celular.
- Altas concentraciones de QUIN se asocian con neurotoxicidad y enfermedades como Huntington o Alzheimer.


## 1. Instalación de dependencias

In [ ]:
!apt -y install openbabel autodock-vina #instalación del toolbox obabel para preparación de proteínas y autodock-vina para docking

!pip -q install py3Dmol biopython pubchempy requests #instalación delibrerias de python
#(py3Dmol para visualización de estructuras, biopython herramientas para bioinformática, pubchempy permite interactuar con la base de PubChem y requests, permite descargar https)

print("Entorno preparado correctamente.")

## 2. Modelado de SNCA desde AlphaFold

Acceder a:
[GoogleColab - AlphaFold2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb#scrollTo=R_AH6JSXaeb2)

Pegar:
[Proteína de prueba - Alpha-synuclein](https://www.uniprot.org/uniprotkb/P37840/entry#names_and_taxonomy)
MDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLYVGSKTKEGVVHGVATVAEKTKEQVTNVGGAVVTGVTAVAQKTVEGAGSIAAATGFVKKDQLGKNEEGAPQEGILEDMPVDPDNEAYEMPSEEGYQDYEPEA


## 2.1 Descargar modelo de GRIN2B desde la base de datos de AlphaFold

Acceder a:
[GRIN2B - AlphaFold](https://alphafold.ebi.ac.uk/entry/Q13224)

Y descargar archivo .pdb

## 2.2 Evaluación de calidad
Las proteínas deben de pasar por un proceso de evaluación de calidad

[SAVES](https://saves.mbi.ucla.edu/)

**Interpretación**

-ERRAT - Análisis estadístico de interacciones atómicas no covalentes en la cadena peptídica (>85)

-WHATCHECK >40 valores de calidad como nomenclaturas, ángulos, átomos faltantes, entre otros.

-Procheck, el Ramachandran mide las distancias y torsiones de átomos en los enlaces peptídicos

## 3. Preparar la proteína receptora para el docking

In [ ]:
# Cargar archivo .pdb de GRIN2B descargado de AlphaFold
#Preparar proteína

!obabel AF-Q13224-F1-model_v6.pdb -O GRIN2B_AF.pdbqt --addH -p 7.4 --partialcharge gasteiger

## 3.1 Limpiar etiquetas erróneas del receptor PDBQT

In [ ]:
receptor_in  = "GRIN2B_AF.pdbqt"
receptor_out = "GRIN2B_AF_clean.pdbqt"

tags = ("ROOT", "ENDROOT", "BRANCH", "ENDBRANCH", "TORSDOF", "CONECT", "MODEL", "ENDMDL")

with open(receptor_in) as fin, open(receptor_out, "w") as fout:
    for line in fin:
        if not line.startswith(tags):
            fout.write(line)

print(f"Archivo limpio guardado como: {receptor_out}")

#Las etiquetas de "tags" solo se usan en ligandos flexibles, no en receptores, que siempre son considerados rígidos

## 4. Descargar el ligando (ácido quinolínico) desde PubChem

Acceder a:
[Ácido quinolínico - Pubchem](https://pubchem.ncbi.nlm.nih.gov/compound/1066)

Y descargar archivo 3D Conformer - SDF

## 4.1 Preparar el ligando para el docking

In [ ]:
!obabel Conformer3D_COMPOUND_CID_1066.sdf -O QUIN.pdbqt -p 7.4 --addH --partialcharge gasteiger

## 5. Definir la región de docking

In [ ]:
# Caja centrada en la cavidad 1.2.49 de ChimeraX: Find Cavities

# Centro medido con 'measure center sel'
center_x, center_y, center_z = 16.49, 6.01, -17.02

# Tamaño estimado de la caja (en Å)
# Basado en el volumen del pocket (~1578 Å³) y margen de exploración.
size_x = size_y = size_z = 28.0

print(f"Center: {center_x:.2f}, {center_y:.2f}, {center_z:.2f}")
print(f"Size  : {size_x:.1f}, {size_y:.1f}, {size_z:.1f}")


## 6. Ejecutar docking con AutoDock Vina

In [ ]:
!vina --receptor GRIN2B_AF_clean.pdbqt --ligand QUIN.pdbqt \
       --center_x {center_x} --center_y {center_y} --center_z {center_z} \
       --size_x {size_x} --size_y {size_y} --size_z {size_z} \
       --exhaustiveness 16 --num_modes 10 \
       --out GRIN2B-QUIN_4.pdbqt | tee log.txt

## 7. Visualizar el complejo proteína–ligando y archivo log.txt

In [ ]:
import re
import pandas as pd

# Transformar log.txt a tabla de energía
modes = []
with open("log.txt") as f:
    for line in f:
        # Líneas típicas en log de vina: "    1         -7.6      0.000      0.000"
        m = re.match(r"\s*(\d+)\s+(-?\d+\.\d+)\s+(\d+\.\d+)\s+(\d+\.\d+)", line)
        if m:
            rank = int(m.group(1))
            affinity = float(m.group(2))  # kcal/mol (más negativo, mejor)
            rmsd_lb = float(m.group(3))
            rmsd_ub = float(m.group(4))
            modes.append((rank, affinity, rmsd_lb, rmsd_ub))

df = pd.DataFrame(modes, columns=["mode", "affinity_kcal_mol", "rmsd_lb", "rmsd_ub"])
if df.empty:
    print("No se pudieron extraer modos del log. Revisa el contenido de log.txt:")
    print(open("log.txt").read())
else:
    from IPython.display import display
    display(df)

# Visualización con Py3Dmol
import py3Dmol

receptor_pdbqt = "GRIN2B_AF_clean.pdbqt"   # para representación cartoon
best_pose_pdbqt = "GRIN2B-QUIN_4.pdbqt"        # salida de Vina

view = py3Dmol.view(width=900, height=600)
view.addModel(open(receptor_pdbqt).read(), "pdb")
view.setStyle({"cartoon": {"color": "green"}})

# Mejor pose del ligando (modelo 1 en out.pdbqt)
view.addModel(open(best_pose_pdbqt).read(), "pdbqt")
view.setStyle({"model": 1}, {"stick": {"color": "orange"}})

# Dibujar la caja del docking (wireframe)
def add_box(view, center, size, color="#00ff00", width=4):
    cx, cy, cz = center
    sx, sy, sz = size
    x0, x1 = cx - sx/2, cx + sx/2
    y0, y1 = cy - sy/2, cy + sy/2
    z0, z1 = cz - sz/2, cz + sz/2
    edges = [
        [(x0,y0,z0),(x1,y0,z0)], [(x0,y1,z0),(x1,y1,z0)], [(x0,y0,z1),(x1,y0,z1)], [(x0,y1,z1),(x1,y1,z1)],
        [(x0,y0,z0),(x0,y1,z0)], [(x1,y0,z0),(x1,y1,z0)], [(x0,y0,z1),(x0,y1,z1)], [(x1,y0,z1),(x1,y1,z1)],
        [(x0,y0,z0),(x0,y0,z1)], [(x1,y0,z0),(x1,y0,z1)], [(x0,y1,z0),(x0,y1,z1)], [(x1,y1,z0),(x1,y1,z1)]
    ]
    for (a,b) in edges:
        view.addLine({
            "start": {'x':a[0],'y':a[1],'z':a[2]},
            "end":   {'x':b[0],'y':b[1],'z':b[2]},
            "color": color, "linewidth": width
        })

add_box(view, (center_x, center_y, center_z), (size_x, size_y, size_z), color="#00ff00", width=4)

view.setBackgroundColor("#000000")
view.zoomTo()
view.show()
view.zoomTo({"model":1})  # centra en el ligando


## 7.1 Medir distancias de interacción

In [ ]:
import numpy as np
from collections import defaultdict

receptor_file = "GRIN2B_AF_clean.pdbqt"   # receptor que usaste en Vina
ligand_file   = "GRIN2B-QUIN_4.pdbqt"     # salida del docking (ajusta si usas otro)

# Leer coordenadas de un PDBQT o PDB
def parse_pdbqt_atoms(path):
    atoms = []
    with open(path) as f:
        for line in f:
            if line.startswith(("ATOM","HETATM")):
                name   = line[12:16].strip()
                resn   = line[17:20].strip()
                chain  = line[21].strip()
                resi   = int(line[22:26])
                x = float(line[30:38]); y = float(line[38:46]); z = float(line[46:54])
                atoms.append({
                    "name": name, "resn": resn, "chain": chain, "resi": resi,
                    "coord": np.array([x,y,z], dtype=float)
                })
    return atoms

# Cargar coordenadas
prot_atoms = parse_pdbqt_atoms(receptor_file)
lig_atoms  = parse_pdbqt_atoms(ligand_file)

# vecinos dentro de 4.0 Å
cutoff = 4.0
contacts = defaultdict(list)

for la in lig_atoms:
    lc = la["coord"]
    for pa in prot_atoms:
        d = np.linalg.norm(lc - pa["coord"])
        if d <= cutoff:
            key = (pa["chain"], pa["resn"], pa["resi"])
            contacts[key].append((la["name"], pa["name"], d))

# Resumen por residuo
rows = []
for (chain,resn,resi), pairs in contacts.items():
    mind = min(d for _,_,d in pairs)
    rows.append((chain, resn, resi, len(pairs), round(mind,2)))

rows = sorted(rows, key=lambda x: (x[4], -x[3]))  # más cercanos primero

import pandas as pd
df_contacts = pd.DataFrame(rows, columns=["chain","resn","resi","n_atom_pairs","min_dist"])
display(df_contacts.head(20))
print(f"Residuos dentro de {cutoff} Å: {len(df_contacts)}")

df_contacts.to_csv("GRIN2B_QUIN_contacts_4A.csv", index=False)
print("Guardado: GRIN2B_QUIN_contacts_4A.csv")
